# Scale Factor Files — Structure and Content

This notebook inspects the correctionlib JSON files stored under `data/corrections/`.
All files are gzip-compressed JSON following the **correctionlib v2 schema** and
are loaded at runtime by `darkbottomline/corrections.py` via `correctionlib.CorrectionSet`.

## Directory layout

```
data/corrections/
├── 2022/          NanoAODv12  (eras CD, EFG)
├── 2023/          NanoAODv12
├── 2024/          NanoAODv15
├── 2025/          NanoAODv15
└── 2026/          (reserved)
```

Each year directory contains:

| File suffix | Contents |
|---|---|
| `_btagging.json.gz` | DeepJet / ParticleNet / RobustParticleTransformer b-tagging SFs and WP values |
| `_muon_HighPt.json.gz` | Muon high-pT ID / isolation SFs |
| `_muon_JPsi.json.gz` | Muon low-pT (J/ψ) SFs |
| `_electron.json.gz` | Electron ID, reconstruction SFs |
| `_electronHlt.json.gz` | Electron HLT trigger SFs |
| `_electronSS_EtDependent.json.gz` | Electron energy scale/smearing (Et-dependent) |
| `_puWeights.json.gz` | Pileup reweighting weights |
| `_jet_jerc.json.gz` | Jet energy resolution / correction (JER/JEC) |
| `_jetid.json.gz` | Jet ID SFs |
| `Cert_Collisions*_Golden.json` | Golden lumi-mask JSON (data only) |

In [ ]:
import sys, gzip, json
from pathlib import Path
import pandas as pd

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

corrections_root = project_root / "data" / "corrections"

## Inventory of all correction files

In [ ]:
rows = []
for year_dir in sorted(corrections_root.iterdir()):
    if not year_dir.is_dir():
        continue
    for f in sorted(year_dir.iterdir()):
        if f.suffix in (".gz", ".json"):
            size_kb = f.stat().st_size / 1024
            rows.append({"Year": year_dir.name, "File": f.name, "Size (KB)": f"{size_kb:.0f}"})

inv = pd.DataFrame(rows)
print(f"Total correction files: {len(inv)}")
inv

## Helper: inspect a correctionlib file

The function below opens a `.json.gz` file and returns a tidy summary of every
correction it contains: name, input variables (name + type), and output variable.

In [ ]:
def inspect_correction_file(path: Path) -> pd.DataFrame:
    """Return a DataFrame summarising all corrections in a correctionlib JSON.gz file."""
    if str(path).endswith(".gz"):
        with gzip.open(path) as fh:
            data = json.load(fh)
    else:
        with open(path) as fh:
            data = json.load(fh)

    rows = []
    for corr in data.get("corrections", []):
        inputs = ", ".join(
            f"{i['name']} ({i['type']})" for i in corr.get("inputs", [])
        )
        output = corr.get("output", {}).get("name", "-")
        rows.append({
            "Correction name": corr["name"],
            "Inputs": inputs,
            "Output": output,
            "Description": (corr.get("description") or "")[:80],
        })
    return pd.DataFrame(rows)

## B-tagging scale factors (`_btagging.json.gz`)

Contains WP threshold values and shape/WP SFs for three b-tagging algorithms.
This analysis uses **DeepJet** at the **medium** working point.

In [ ]:
btag_file = corrections_root / "2022" / "Run3-22CDSep23-Summer22-NanoAODv12_btagging.json.gz"
btag_df = inspect_correction_file(btag_file)
print(f"File: {btag_file.name}")
print(f"Corrections: {len(btag_df)}")
btag_df

In [ ]:
# Show the WP threshold values for DeepJet
with gzip.open(btag_file) as fh:
    btag_data = json.load(fh)

wp_correction = next(c for c in btag_data["corrections"] if c["name"] == "deepJet_wp_values")

print("deepJet_wp_values — working point thresholds:")
print(f"  Inputs: {[i['name'] for i in wp_correction['inputs']]}")
print()

# Navigate into the data node to extract WP → discriminant value
def extract_category_values(node):
    """Recursively extract {key: value} from a Category/Transform/Value data node."""
    if node.get("nodetype") == "category":
        result = {}
        for key_val in node.get("keys", []):
            k = key_val
            # find matching content
        # Try to get keys and content as parallel lists
        keys = node.get("keys", [])
        content = node.get("content", [])
        return dict(zip(keys, content))
    return node

node = wp_correction.get("data", {})
wps = extract_category_values(node)
for wp_name, val in wps.items():
    if isinstance(val, dict):
        inner_val = val.get("value", val)
    else:
        inner_val = val
    print(f"  {wp_name}: {inner_val}")

## Muon scale factors (`_muon_HighPt.json.gz`)

High-pT muon ID and isolation scale factors. Inputs are `eta`, `pt` (or `p` for global muons),
and a `scale_factors` string selecting `nominal`, `systup`, or `systdown`.

In [ ]:
muon_file = corrections_root / "2022" / "Run3-22CDSep23-Summer22-NanoAODv12_muon_HighPt.json.gz"
muon_df = inspect_correction_file(muon_file)
print(f"File: {muon_file.name}")
print(f"Corrections: {len(muon_df)}")
muon_df

In [ ]:
# Show how to evaluate a muon SF with correctionlib
try:
    from correctionlib import CorrectionSet
    import tempfile, os

    # correctionlib needs an uncompressed file or direct JSON string
    with gzip.open(muon_file) as fh:
        raw = fh.read()

    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
        tmp.write(raw)
        tmp_path = tmp.name

    cset = CorrectionSet.from_file(tmp_path)
    os.unlink(tmp_path)

    print("Corrections available:", list(cset.keys()))
    print()

    # Evaluate HighPtID SF for a muon with eta=0.5, pt=100 GeV
    corr_name = "NUM_HighPtID_DEN_GlobalMuonProbes"
    sf = cset[corr_name].evaluate(0.5, 100.0, "nominal")
    sf_up = cset[corr_name].evaluate(0.5, 100.0, "systup")
    sf_dn = cset[corr_name].evaluate(0.5, 100.0, "systdown")
    print(f"{corr_name}")
    print(f"  eta=0.5, pt=100 GeV  →  nominal={sf:.4f}, up={sf_up:.4f}, down={sf_dn:.4f}")

except ImportError:
    print("correctionlib not installed — install with: pip install correctionlib")
except Exception as e:
    print(f"Error evaluating correction: {e}")

## Electron scale factors (`_electron.json.gz`)

Contains the unified `Electron-ID-SF` correction for ID and reconstruction.
Inputs are `year`, `ValType` (sf/sfup/sfdown), `WorkingPoint`, `eta`, and `pt`.

In [ ]:
ele_file = corrections_root / "2022" / "Run3-22CDSep23-Summer22-NanoAODv12_electron.json.gz"
ele_df = inspect_correction_file(ele_file)
print(f"File: {ele_file.name}")
print(f"Corrections: {len(ele_df)}")
ele_df

In [ ]:
# Show working points available in the electron SF
with gzip.open(ele_file) as fh:
    ele_data = json.load(fh)

ele_corr = ele_data["corrections"][0]
print(f"Correction: {ele_corr['name']}")
print(f"Inputs: {[(i['name'], i['type']) for i in ele_corr['inputs']]}")
print()

# Drill into the data tree to find WorkingPoint categories
def find_categories(node, depth=0, path=""):
    """Recursively print category node keys."""
    if not isinstance(node, dict):
        return
    if node.get("nodetype") == "category":
        keys = node.get("keys", [])
        print(f"{'  '*depth}Category @ '{path}': {keys}")
        for key, child in zip(node.get("keys", []), node.get("content", [])):
            find_categories(child, depth + 1, f"{path}/{key}")
    else:
        for sub_key in ["content"]:
            child = node.get(sub_key)
            if isinstance(child, dict):
                find_categories(child, depth, path)
            elif isinstance(child, list):
                for item in child[:1]:  # Only first to avoid large output
                    if isinstance(item, dict):
                        find_categories(item, depth, path)

find_categories(ele_corr.get("data", {}), path="root")

## Pileup reweighting (`_puWeights.json.gz`)

One correction per run era. Input is `NumTrueInteractions` (from `Pileup_nTrueInt`)
and a `weights` string: `nominal`, `up`, or `down`.

In [ ]:
pu_file = corrections_root / "2022" / "Run3-22CDSep23-Summer22-NanoAODv12_puWeights.json.gz"
pu_df = inspect_correction_file(pu_file)
print(f"File: {pu_file.name}")
print(f"Corrections: {len(pu_df)}")
pu_df

In [ ]:
# Plot the pileup weight vs NumTrueInteractions for the nominal scenario
import numpy as np

try:
    import matplotlib.pyplot as plt
    from correctionlib import CorrectionSet
    import tempfile, os

    with gzip.open(pu_file) as fh:
        raw = fh.read()
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
        tmp.write(raw)
        tmp_path = tmp.name
    cset = CorrectionSet.from_file(tmp_path)
    os.unlink(tmp_path)

    corr_name = list(cset.keys())[0]
    npu = np.linspace(0, 80, 200)

    w_nom = np.array([cset[corr_name].evaluate(x, "nominal") for x in npu])
    w_up  = np.array([cset[corr_name].evaluate(x, "up")      for x in npu])
    w_dn  = np.array([cset[corr_name].evaluate(x, "down")    for x in npu])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(npu, w_nom, label="nominal", color="black")
    ax.fill_between(npu, w_dn, w_up, alpha=0.3, label="up/down unc.", color="steelblue")
    ax.set_xlabel("NumTrueInteractions")
    ax.set_ylabel("Pileup weight")
    ax.set_title(f"{corr_name}")
    ax.legend()
    plt.tight_layout()
    plt.show()

except ImportError as e:
    print(f"Missing dependency: {e} — install with: pip install correctionlib matplotlib")
except Exception as e:
    print(f"Error: {e}")

## Jet energy corrections (`_jet_jerc.json.gz`)

Contains JEC (Jet Energy Correction) and JER (Jet Energy Resolution) corrections.
These are used to shift and smear jet pT before applying the jet selection.

In [ ]:
jec_file = corrections_root / "2022" / "Run3-22CDSep23-Summer22-NanoAODv12_jet_jerc.json.gz"
jec_df = inspect_correction_file(jec_file)
print(f"File: {jec_file.name}")
print(f"Corrections: {len(jec_df)}")
for i in range(len(jec_df["Correction name"])):
    if "L1L2L3Res" in str(jec_df["Correction name"][i]):
        print(jec_df["Correction name"][i])
    else:
        continue


## 2025 corrections (NanoAODv15)

The 2025 directory has a reduced set of files (electron ID uses the `_electronSS_EtDependent` format
and there is no separate pileup or muon JPsi file yet).

In [ ]:
print("2025 correction files:")
for f in sorted((corrections_root / "2025").iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:60s}  {size_kb:6.0f} KB")
print()

# Inspect 2025 muon HighPt SFs
muon_2025 = corrections_root / "2025" / "Run3-25Prompt-Summer24-NanoAODv15_muon_HighPt.json.gz"
if muon_2025.exists():
    df_2025 = inspect_correction_file(muon_2025)
    print(f"Muon HighPt (2025) — {len(df_2025)} corrections:")
    print(df_2025.to_string(index=False))

## How corrections are applied in the framework

At runtime, `darkbottomline/corrections.py` loads each year's files into a `CorrectionSet`
and wraps evaluation in helper functions. The pattern is:

```python
from correctionlib import CorrectionSet

cset = CorrectionSet.from_file("Run3-22CDSep23-Summer22-NanoAODv12_btagging.json.gz")

# B-tagging WP value
wp_cut = cset["deepJet_wp_values"].evaluate("M")

# B-tagging SF for a jet
sf = cset["deepJet_comb"].evaluate("central", "M", hadron_flavour, abs_eta, pt)

# Pileup weight
pu_w = cset["Collisions2022_..."].evaluate(nTrueInt, "nominal")

# Muon high-pT ID SF
mu_sf = cset["NUM_HighPtID_DEN_GlobalMuonProbes"].evaluate(eta, pt, "nominal")

# Electron ID SF
el_sf = cset["Electron-ID-SF"].evaluate("2022Re-recoBCD", "sf", "Tight", eta, pt)
```

Systematics are handled by evaluating with `"systup"` / `"systdown"` (muon)
or `"sfup"` / `"sfdown"` (electron) and taking the ratio to the nominal weight.

## Golden JSON lumi mask

The Golden JSON is a plain (uncompressed) JSON file mapping run numbers to
lists of good lumi-block ranges. It is applied to data events only.

In [ ]:
golden_file = corrections_root / "2022" / "Cert_Collisions2022_355100_362760_Golden.json"

with open(golden_file) as fh:
    golden = json.load(fh)

run_numbers = [int(r) for r in golden.keys()]
total_lumis = sum(len(v) for v in golden.values())

print(f"File: {golden_file.name}")
print(f"  Run range : {min(run_numbers)} – {max(run_numbers)}")
print(f"  Good runs : {len(run_numbers)}")
print(f"  Lumi-block ranges: {total_lumis}")
print()
print("First 5 runs:")
for run, lumis in list(golden.items())[:5]:
    print(f"  Run {run}: {lumis[:3]}{'...' if len(lumis) > 3 else ''}")